In [38]:
import os
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
client=Groq()

In [39]:
client

In [40]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
# from google.colab import userdata

c:\Agile Project\ProjectBuild3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
pdf_location='data/raw'
loader=PyPDFDirectoryLoader(pdf_location)

In [8]:
type(loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [20]:

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
encoding_name='cl100k_base',
chunk_size=400,
chunk_overlap=25
)

In [21]:
chunks = loader.load_and_split(text_splitter)

In [22]:
len(chunks)

236

In [25]:
print(chunks[1])

page_content='Dear Shareholders:
When I graduated from college, I wanted to be a sportscaster. After sending my resume reel to many small
markets around the U.S., and only getting two nibbles, I settled on doing sports production at a major network.
To make extra money, I also coached my former high school soccer team, and worked at a retail golf store.
Six months later, a college classmate convinced me to interview at the consumer products company where he
worked, and I spent three years as a Product Manager there. I left that job to try some of my own businesses.
After deciding these businesses weren’t my calling, I tried short stints in sales and investment banking,
before going back to graduate school and ending up at Amazon three days after my last final exam in
May 1997.
Not exactly a straight line.
AWS followed lots of squiggly lines, too. The original vision included storage, compute, payments, and
human intelligence. Some of those (e.g. storage and compute) became lynchpins in

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_4604\1750141895.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [18]:
amazon_collection = 'amazon-2025'

In [15]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [16]:
chromadb_client.heartbeat()

1781243915821663000

In [19]:
vectorstore = Chroma(
    collection_name=amazon_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [26]:
i = 0 

while i < len(chunks): 
    vectorstore.add_documents( 
        documents=chunks[i:i+25], 
        ids=["text_" + str(i) for i in range(i, i+25)] 
    )

    i += 25
    time.sleep(5) 

In [27]:
vectorstore_persisted = Chroma(
    collection_name=amazon_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [28]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [29]:
user_query = "What are the key financial highlights for Amazon in 2025?"

In [33]:
relevant_chunks=retriever.invoke(user_query)

In [34]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [35]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [36]:
i = 0
for document in relevant_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
Guidance
We provided guidance on February 5, 2026, in our earnings release furnished on Form 8-K as set forth below. These 
forward-looking statements reflect Amazon.com’s expectations as of February 5, 2026, and are subject to substantial 
uncertainty. Our results are inherently unpredictable and may be materially affected by many factors, such as fluctuations in 
foreign exchange rates and energy prices, changes in global economic and geopolitical conditions, tariff and trade policies, 
resource and supply volatility, including for memory chips, and customer demand and spending (including the impact of 
recessionary fears), inflation, interest rates, regional labor market constraints, world events, the rate of growth of the internet, 
online commerce, cloud services, and new and emerging technologies, as well as those outlined in Item 1A of Part I, “Risk 
Factors.” 
First Quarter 2026 Guidance
• Net sales are expected to be between $173.5 billion and $178.5 bill

In [41]:

model_name = 'openai/gpt-oss-120b'

relevant_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

**Amazon’s 2025 financial highlights**

- **Total revenue:** $717 billion, a 12% year‑over‑year increase from $638 billion in 2024.  
- **Segment revenue:**  
  - North America – $426 billion (up 10% YoY)  
  - International – $162 billion (up 13% YoY)  
  - AWS – $129 billion (up 20% YoY)  
- **Operating income:** $80 billion, a 17% YoY rise, delivering an operating margin of 11.2% (up from 10.8% in 2024).  
- **Consolidated net sales:** $716.9 billion (product sales $296.3 billion + service sales $420.7 billion).  
- **Free cash flow:** $11 billion, down from $38 billion in 2024, primarily due to $50.7 billion of capital expenditures for AI‑related investments.


In [75]:
# Import prompts from prompts.py and ensure the latest version is loaded
import importlib
import prompts
importlib.reload(prompts)
from prompts import query_classification_prompt, answer_generation_prompt, query_message_template
from output_parser import parse_classification, parse_answer
import json
from datetime import datetime


In [70]:

# Step 1: Classify the user query using prompts.py
classification_prompt = query_classification_prompt(user_query)
print("Classification Prompt:")
print(classification_prompt)


Classification Prompt:
You are a helpful assistant for classifying user queries into one of the following categories: 
FACTUAL_LOOKUP
SUMMARY
COMPARISON
RISK_ANALYSIS
UNANSWERABLE_OR_SPECULATIVE
FOLLOW_UP
OTHER.
You must classify the user query before providing an answer. If the query does not fit into any of the above categories, classify it as "Other".
Classify the following user query: What are the key financial highlights for Amazon in 2025?

Expected Output Format:
{
"query_type": "",
"requires_retrieval": true,
"requires_comparison": false,
"answer_style": "",
"reasoning_summary": ""
}



In [71]:
# Classify the query with Groq API
classification_messages = [
    {"role": "system", "content": classification_prompt},
    {"role": "user", "content": query_message_template.format(user_query)}
]

try:
    classification_response = client.chat.completions.create(
        model=model_name,
        messages=classification_messages,
        temperature=0
    )
    classification_result = classification_response.choices[0].message.content.strip()
    print("Query Classification:")
    print(classification_result)
except Exception as e:
    classification_result = f"Error: {e}"
    print(classification_result)


Query Classification:
{
  "query_type": "FACTUAL_LOOKUP",
  "requires_retrieval": true,
  "requires_comparison": false,
  "answer_style": "summary",
  "reasoning_summary": "The user asks for specific, factual information about Amazon's financial performance in 2025, which requires retrieving up‑to‑date financial data. No comparison or analysis is requested, so it is classified as a factual lookup."
}


In [74]:

# Step 2: Generate answer using prompts.py
answer_prompt = answer_generation_prompt(user_query, context_for_query)
print("\nAnswer Generation Prompt:")
print(answer_prompt)



Answer Generation Prompt:
You are a helpful assistant for answering user queries based on the provided context.
Rules:
1. The answer must be based only on retrieved context.
2. Every important claim must map to a retrieved source.
3. The assistant must not invent unsupported information.
4. If retrieved context is weak, answerability should be PARTIALLY_ANSWERED or 
NOT_FOUND.
5. If the question is speculative, the answer should refuse speculation.


Answer the following user query based on the provided context: What are the key financial highlights for Amazon in 2025?
Context: Guidance
We provided guidance on February 5, 2026, in our earnings release furnished on Form 8-K as set forth below. These 
forward-looking statements reflect Amazon.com’s expectations as of February 5, 2026, and are subject to substantial 
uncertainty. Our results are inherently unpredictable and may be materially affected by many factors, such as fluctuations in 
foreign exchange rates and energy prices, chan

In [68]:
# Generate answer with Groq API
answer_messages = [
    {"role": "system", "content": answer_prompt},
    {"role": "user", "content": user_query}
]

try:
    answer_response = client.chat.completions.create(
        model=model_name,
        messages=answer_messages,
        temperature=0
    )
    answer_result = answer_response.choices[0].message.content.strip()
    print("Generated Answer:")
    print(answer_result)
except Exception as e:
    answer_result = f"Error: {e}"
    print(answer_result)


Generated Answer:
{
  "answer": "In 2025 Amazon delivered a strong financial performance:\n- Total revenue (net sales) reached **$717 billion**, up **12% YoY** from $638 billion in 2024. 【Amazon’s revenue in 2025 grew 12% year‑over‑year (“Y oY”) from $638 billion to $717 billion.】\n- By segment, revenue was **$426 billion** in North America (up 10% YoY), **$162 billion** internationally (up 13% YoY), and **$129 billion** for AWS (up 20% YoY). 【North America revenue increased 10% Y oY from $387 billion to $426 billion, International revenue grew 13% Y oY from $143 billion to $162 billion, and AWS revenue increased 20% Y oY, from $108 billion to $129 billion.】\n- Operating income rose to **$80 billion**, a **17% increase** over the prior year, improving the operating margin to **11.2%** (from 10.8%). 【Amazon’s operating income in 2025 improved 17% Y oY, from $69 billion (an operating margin of 10.8%) to $80 billion (an operating margin of 11.2%).】\n- Free cash flow fell sharply to **$11 

In [76]:
# Step 3: Save results to outputs folder
import os

# Create outputs folder if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Parse results into structured JSON schema
classification_data = parse_classification(classification_result)
answer_data = parse_answer(answer_result)

results = {
    "question": user_query,
    "query_type": classification_data.get("query_type", ""),
    "answerability": answer_data.get("answerability", "NOT_FOUND"),
    "confidence": answer_data.get("confidence", "LOW"),
    "answer": answer_data.get("answer", ""),
    "supporting_evidence": answer_data.get("supporting_evidence", []),
    "sources": answer_data.get("sources", []),
    "classification": classification_data,
    "context_sources": len(relevant_chunks)
}

# Save as JSON
output_filename = f"outputs/query_response_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Results saved to {output_filename}")

# Also save as text file for easy reading
txt_filename = f"outputs/query_response_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(txt_filename, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("QUERY CLASSIFICATION & ANSWER GENERATION REPORT\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Timestamp: {datetime.now().isoformat()}\n")
    f.write(f"User Query: {user_query}\n\n")
    f.write("-" * 70 + "\n")
    f.write("CLASSIFICATION:\n")
    f.write("-" * 70 + "\n")
    f.write(json.dumps(classification_data, indent=2))
    f.write("\n\n" + "-" * 70 + "\n")
    f.write("ANSWER:\n")
    f.write("-" * 70 + "\n")
    f.write(json.dumps(answer_data, indent=2))
    f.write("\n\n" + "-" * 70 + "\n")
    f.write(f"Context Sources Used: {len(relevant_chunks)}\n")

print(f"✓ Results saved to {txt_filename}")

# Display summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(json.dumps(results, indent=2))



✓ Results saved to outputs/query_response_20260612_135150.json
✓ Results saved to outputs/query_response_20260612_135150.txt

SUMMARY
{
  "question": "What are the key financial highlights for Amazon in 2025?",
  "query_type": "FACTUAL_LOOKUP",
  "answerability": "ANSWERED",
  "confidence": "HIGH",
  "answer": "In 2025 Amazon delivered a strong financial performance:\n- Total revenue (net sales) reached **$717\u202fbillion**, up **12% YoY** from $638\u202fbillion in 2024.\u202f\u3010Amazon\u2019s revenue in 2025 grew 12% year\u2011over\u2011year (\u201cY oY\u201d) from $638 billion to $717 billion.\u3011\n- By segment, revenue was **$426\u202fbillion** in North America (up 10% YoY), **$162\u202fbillion** internationally (up 13% YoY), and **$129\u202fbillion** for AWS (up 20% YoY).\u202f\u3010North America revenue increased 10% Y oY from $387\u202fbillion to $426\u202fbillion, International revenue grew 13% Y oY from $143\u202fbillion to $162\u202fbillion, and AWS revenue increased 20%